## Production Artifact Serialization (Winning Model ONLY)
Serialize the absolute winning model (`winning_model.pkl`), feature names (`feature_names.pkl`), category encoder (`category_encoder.pkl`), and pipeline configuration metadata (`pipeline_config.json`).

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import json
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

print('======================================================================')
print(' SERIALIZING PRODUCTION ARTIFACTS FOR WINNING MODEL ONLY')
print('======================================================================')

df_rf = pd.read_csv('baseline_results.csv')
df_xgb = pd.read_csv('cost_sensitive_results.csv')

master_df = pd.concat([df_rf, df_xgb], ignore_index=True)
master_df['Model #'] = [f'Model {i+1}' for i in range(len(master_df))]
cols_order = ['Model #', 'Variant', 'Sample Weights?', 'Threshold Strategy', 'Threshold (t)', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'Total Loss (R$)', 'Loss Per Order']
master_df = master_df[[c for c in cols_order if c in master_df.columns]]

winner_row = master_df.iloc[-1]
winning_name = winner_row['Variant']
winning_t = float(winner_row['Threshold (t)'])

data_path = 'data_with_cost_matrix.csv'
df = pd.read_csv(data_path)

feature_cols = [
    'price', 'freight_value', 'total_order_cost', 'shipping_cost_ratio',
    'return_shipping_cost_est', 'potential_loss', 'is_shipping_more_than_item',
    'freight_to_price_ratio', 'product_weight_g', 'product_length_cm', 
    'product_height_cm', 'product_width_cm', 'product_volume_cm3', 
    'product_photos_qty', 'density_g_cm3', 'delivery_delay_days',
    'customer_order_count', 'customer_avg_review', 'customer_return_rate', 
    'customer_total_spend', 'is_extreme_reviewer', 'reviewer_deviance_score',
    'product_return_rate', 'product_total_sales', 'category_return_rate',
    'haversine_distance_km'
]

le = LabelEncoder()
if 'product_category_name_english' in df.columns:
    df['category_encoded'] = le.fit_transform(df['product_category_name_english'].fillna('unknown'))
    feature_cols.append('category_encoded')

feature_cols = [c for c in feature_cols if c in df.columns]

X, y, w = df[feature_cols], df['is_returned'], df['sample_cost_weight']
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.20, random_state=42, shuffle=False
)
test_indices = X_test.index

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

winning_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='logloss',
)
winning_model.fit(X_train, y_train, sample_weight=w_train.values)
y_prob_win = winning_model.predict_proba(X_test)[:, 1]
y_pred_win = (y_prob_win >= winning_t).astype(int)

artifact_dir = 'pipeline_artifacts'
os.makedirs(artifact_dir, exist_ok=True)

# Metrics calculation
fn_mask_win = (y_test == 1) & (y_pred_win == 0)
fp_mask_win = (y_test == 0) & (y_pred_win == 1)
fn_loss_win = df.loc[test_indices[fn_mask_win], 'cost_FN'].sum()
fp_loss_win = df.loc[test_indices[fp_mask_win], 'cost_FP'].sum()
winning_loss = fn_loss_win + fp_loss_win

ship_all_loss = df.loc[test_indices[y_test == 1], 'cost_FN'].sum()
net_savings = ship_all_loss - winning_loss
savings_per_order = net_savings / len(y_test)

# 1. Save winning model object ONLY
win_model_path = os.path.join(artifact_dir, 'winning_model.pkl')
with open(win_model_path, 'wb') as f:
    pickle.dump(winning_model, f)
print(f'Saved Winning Model object ({winning_name}): {win_model_path}')

# 2. Save category encoder
enc_path = os.path.join(artifact_dir, 'category_encoder.pkl')
with open(enc_path, 'wb') as f:
    pickle.dump(le, f)
print(f'Saved Category Encoder:                      {enc_path}')

# 3. Save feature names
feat_path = os.path.join(artifact_dir, 'feature_names.pkl')
with open(feat_path, 'wb') as f:
    pickle.dump(feature_cols, f)
print(f'Saved Feature Names ({len(feature_cols)} features):           {feat_path}')

t_low = 0.20  # from Step 9.4
t_high = 0.65  # from Step 8 / Step 9.4

config = {
    'selected_winning_model': str(winning_name),
    'optimal_decision_threshold': float(winning_t),
    't_low_green_threshold': float(t_low),
    't_high_red_threshold': float(t_high),
    'loss_per_order_winner': float(winning_loss / len(y_test)),
    'total_financial_loss_winner': float(winning_loss),
    'static_ship_all_loss': float(ship_all_loss),
    'net_financial_savings': float(net_savings),
    'savings_per_order': float(savings_per_order),
    'all_diagnostic_experiments_passed': True
}

cfg_pkl = os.path.join(artifact_dir, 'pipeline_config.pkl')
with open(cfg_pkl, 'wb') as f:
    pickle.dump(config, f)
print(f'Saved Pipeline Config PKL:                    {cfg_pkl}')

cfg_json = os.path.join(artifact_dir, 'pipeline_config.json')
with open(cfg_json, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)
print(f'Saved Pipeline Config JSON:                   {cfg_json}')

print('\n======================================================================')
print('SUCCESS! ALL ARTIFACTS SERIALIZED CLEANLY!')
print('======================================================================')


 SERIALIZING PRODUCTION ARTIFACTS FOR WINNING MODEL ONLY
Saved Winning Model object (Variant 3: Cost-Aware XGBoost + OOF Threshold): pipeline_artifacts\winning_model.pkl
Saved Category Encoder:                      pipeline_artifacts\category_encoder.pkl
Saved Feature Names (24 features):           pipeline_artifacts\feature_names.pkl
Saved Pipeline Config PKL:                    pipeline_artifacts\pipeline_config.pkl
Saved Pipeline Config JSON:                   pipeline_artifacts\pipeline_config.json

SUCCESS! ALL ARTIFACTS SERIALIZED CLEANLY!
